<a href="https://colab.research.google.com/github/siddhartha-sai-17/Celebal-Excellence-Internship-/blob/main/week8%3CB_Sai_Siddhartha%3E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 8 Assignment: Single-Agent Task-Routing Pipeline

**Objective:** Build a simple agent that routes an incoming query to the
correct tool based on keyword pattern-matching, and returns a clean,
structured JSON response.

**Pipeline covered in this notebook:**
1. Baseline tools: a safe math `calculator` and a simple `extract_keywords`
   function
2. `agent(query)` — the task-routing pipeline that decides which tool (if
   any) should handle a given query
3. Structured JSON output: `{"type": ..., "result": ...}`
4. Automated validation checks against a fixed array of test queries
5. An interactive `while True` loop for manual testing


## 1. Baseline Tools

These are the two "tools" the agent can call. They're written to be safe and
dependency-free:

- **`calculator(expression)`** evaluates a basic arithmetic expression using
  Python's `ast` module rather than raw `eval()`, so it only ever supports
  numbers and `+ - * / // % **` — it can't execute arbitrary code.
- **`extract_keywords(text, top_n=5)`** does simple frequency-based keyword
  extraction: lowercase, strip punctuation, drop a small stopword list, and
  return the most common remaining words.


In [1]:
import ast
import operator
import re
import json
from collections import Counter

# --- Tool 1: safe calculator -------------------------------------------------
_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,
    ast.UAdd: operator.pos,
}

def _eval_node(node):
    if isinstance(node, ast.Constant):  # Python 3.8+
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numeric constants are allowed.")
    elif isinstance(node, ast.BinOp):
        op_func = _OPS.get(type(node.op))
        if op_func is None:
            raise ValueError(f"Operator {type(node.op).__name__} is not allowed.")
        return op_func(_eval_node(node.left), _eval_node(node.right))
    elif isinstance(node, ast.UnaryOp):
        op_func = _OPS.get(type(node.op))
        if op_func is None:
            raise ValueError(f"Operator {type(node.op).__name__} is not allowed.")
        return op_func(_eval_node(node.operand))
    else:
        raise ValueError(f"Unsupported expression element: {type(node).__name__}")

def calculator(expression):
    """Safely evaluate a basic arithmetic expression string."""
    tree = ast.parse(expression, mode="eval")
    return _eval_node(tree.body)


# --- Tool 2: keyword extractor ------------------------------------------------
_STOPWORDS = set("""
a an the is are was were be been being of to in on at for with and or but if
then so as by from this that these those it its i you he she we they what
which who whom will would can could should shall may might must do does did
not no yes there here about into over under again further once
""".split())

def extract_keywords(text, top_n=5):
    """Return the top_n most frequent non-stopword tokens in the text."""
    words = re.findall(r"[a-zA-Z']+", text.lower())
    filtered = [w for w in words if w not in _STOPWORDS and len(w) > 2]
    counts = Counter(filtered)
    return [word for word, _ in counts.most_common(top_n)]


# Quick smoke tests for the baseline tools
print(calculator("12 + 8 * 2"))
print(extract_keywords("Machine learning models require large amounts of quality training data to perform well"))


28
['machine', 'learning', 'models', 'require', 'large']


## 2. Task-Routing Agent

`agent(query)` inspects the lowercased query string and routes it with
explicit conditional checks:

- Contains **"calculate"** → extract the math expression from the query and
  send it to `calculator`
- Contains **"keywords"** → strip the trigger word and send the remaining
  text to `extract_keywords`
- Anything else → a fallback general text response
- Any failure along the way (bad expression, empty input, etc.) → an
  `"error"` response instead of a crash

The response is always a JSON-serializable dict of the form
`{"type": "calculation" | "keywords" | "general" | "error", "result": ...}`.


In [2]:
def _extract_math_expression(query):
    """Pull the arithmetic portion out of a query like 'calculate 5 + 3 * 2'."""
    cleaned = re.sub(r"\bcalculate\b", "", query, flags=re.IGNORECASE)
    match = re.search(r"[\d\.\+\-\*/%\(\)\s]+", cleaned)
    if not match or not match.group().strip():
        raise ValueError("No valid mathematical expression found in query.")
    return match.group().strip()


def agent(query):
    """Route a query to the correct tool and return a structured JSON-able dict."""
    if not isinstance(query, str) or not query.strip():
        return {"type": "error", "result": "Query must be a non-empty string."}

    q_lower = query.lower()

    try:
        if "calculate" in q_lower:
            expression = _extract_math_expression(query)
            result = calculator(expression)
            return {"type": "calculation", "result": result}

        elif "keywords" in q_lower:
            text = re.sub(r"\bkeywords\b", "", query, flags=re.IGNORECASE).strip()
            if not text:
                return {"type": "error", "result": "No text provided for keyword extraction."}
            result = extract_keywords(text)
            return {"type": "keywords", "result": result}

        else:
            return {"type": "general", "result": f"You said: {query}"}

    except ZeroDivisionError:
        return {"type": "error", "result": "Division by zero is not allowed."}
    except Exception as e:
        return {"type": "error", "result": f"Could not process query: {e}"}


## 3. Automated Validation

We run the agent against a fixed array of test queries covering all four
response types (`calculation`, `keywords`, `general`, `error`) and print each
result as a JSON block, so retrieval/routing accuracy can be checked at a
glance.


In [3]:
test_queries = [
    "calculate 12 + 8 * 2",
    "calculate (10 - 4) / 3",
    "please calculate 9 / 0",                    # error: division by zero
    "calculate abc + 2",                         # error: invalid expression
    "give me keywords from: Machine learning models require large amounts of quality training data to perform well",
    "keywords",                                  # error: no text supplied
    "hello, how are you today?",                 # general fallback
    "",                                           # error: empty query
]

print("=== Automated validation ===")
for q in test_queries:
    response = agent(q)
    print(f"Query: {q!r}")
    print(json.dumps(response, indent=2))
    print("-" * 60)


=== Automated validation ===
Query: 'calculate 12 + 8 * 2'
{
  "type": "calculation",
  "result": 28
}
------------------------------------------------------------
Query: 'calculate (10 - 4) / 3'
{
  "type": "calculation",
  "result": 2.0
}
------------------------------------------------------------
Query: 'please calculate 9 / 0'
{
  "type": "error",
  "result": "Division by zero is not allowed."
}
------------------------------------------------------------
Query: 'calculate abc + 2'
{
  "type": "error",
  "result": "Could not process query: No valid mathematical expression found in query."
}
------------------------------------------------------------
Query: 'give me keywords from: Machine learning models require large amounts of quality training data to perform well'
{
  "type": "keywords",
  "result": [
    "give",
    "machine",
    "learning",
    "models",
    "require"
  ]
}
------------------------------------------------------------
Query: 'keywords'
{
  "type": "error",
  

## 4. Interactive Testing Loop

Run the cell below to manually test the agent. Type `exit` or `quit` to stop.


In [4]:
while True:
    user_input = input("Ask something (or type 'exit' to quit): ")
    if user_input.strip().lower() in ("exit", "quit"):
        print("Session ended.")
        break
    response = agent(user_input)
    print(json.dumps(response, indent=2))


Ask something (or type 'exit' to quit): who is the captain of indian team in t20i
{
  "type": "general",
  "result": "You said: who is the captain of indian team in t20i"
}
Ask something (or type 'exit' to quit): exit
Session ended.


## 5. Analysis & Observations

*(Replace this placeholder with your own written observations before submitting.)*

- **Routing accuracy:** Did every test query above land in the correct
  `"type"` bucket? Point to any surprising routing decisions.

  The agent correctly identified most user queries and sent them to the correct function. For example, calculation questions were sent to the calculator, while text-related questions were sent to the keyword extraction function. The routing worked well for simple and clear inputs.




- **Error handling:** Look at the division-by-zero and invalid-expression
  cases — did the agent fail safely with a clear error message instead of
  crashing?

  The agent handled incorrect inputs properly. For example, if a user tried to divide by zero or entered an invalid mathematical expression, the program displayed a clear error message instead of crashing.

- **Ambiguous queries:** What happens with a query that contains both
  "calculate" and "keywords"? Is that the routing behavior you'd want, or
  would you change the priority order?

  Some queries can have more than one meaning. In such cases, the agent chooses the first matching function based on its predefined rules. This works, but it may not always understand the user's actual intention.

- **Limitations:** The keyword extractor is purely frequency-based with a
  small stopword list — where does that break down (e.g. short texts,
  domain-specific jargon)?

  The agent depends on predefined keywords to decide what to do. If the user asks the same question using different words, the agent may not always choose the correct function. It also cannot remember previous conversations or understand context like advanced AI agents.


- **Key takeaway:** In your own words, what's the difference between this
  simple keyword-routed agent and a more general "stateful directed graph"
  style agent pipeline (e.g. LangGraph), where routing can depend on
  accumulated state across multiple steps rather than a single query string?


  This experiment showed how an AI agent can automatically choose the correct task based on the user's input. It also demonstrated the importance of routing, handling errors, and organizing different functions into one pipeline. While this rule-based agent works well for simple tasks, more advanced frameworks like LangGraph can remember previous steps and handle more complex conversations.